# Name That LLM — submission notebook

18-class attribution. **Offline by design**: no network calls, no downloads, no API keys.

## Current candidate: cap-40k pool-MIX blend (D028/D037)

`0.5 × char_tfidf_logreg + 0.5 × style_only`, blended on row-wise z-normalised margins.
`char_tfidf_logreg` is fit on the official train **plus the globally decontaminated
external pool** (Open Pool v15, CC BY-SA 4.0): the pool is screened **once** against the
full official train (226,191 → 226,049 clean rows), then capped at 40,000. That keep-mask
is fold-independent, so this is **not** the transductive per-fold route D026 disabled.
`style_only` stays official-only — the pool does not help it.

Local grouped-OOF balanced accuracy (**CV, not a leaderboard score**): **0.4427** seed-42,
**0.4304** seed-13, **0.4426** nested outer-fold gate; official-only control **0.3760**.
Reproduced independently from identity-verified OOF by `experiments/blend_deberta.py`
(D039). The local test file has only three example rows, so its prediction is a smoke
check, not a leaderboard submission.

```
python -m src.cli predict --config configs/submission.yaml --data-dir data --fit \
  --pool artifacts/pool_audit/ntl_open_pool.parquet --pool-models char_tfidf_logreg --pool-cap 40000
python -m src.cli package --dataset official --out artifacts/kaggle_dataset --kaggle-user <you>
```

## Safe fallback: the last *confirmed* leaderboard submission

The **official-only 0.5 char + 0.5 style argmax blend** (D009/D012), public LB **0.356**,
remains the confirmed baseline (grouped OOF 0.3760 seed-42 / 0.3582 seed-13). Rebuild it by
omitting `--pool`/`--pool-models`. **Submitting is owner-only** (AGENTS.md): this repo
prepares the artifacts and the command, but never runs `kaggle competitions submit`, and a
higher grouped-CV score is not a leaderboard result until the owner submits it. The
seed-2027 subset is **examined**, not an untouched gate (D035).

> **Crossed-design inference (D016) is WITHDRAWN (D017/D018).** Its gain was a
> k=18 artifact; at hypothetical k≈2 the Hungarian paired gain is marginal and unreliable.
> The notebook rejects crossed inference.

Inputs required on the right-hand panel:
1. the competition dataset (`name-that-llm`), and
2. the prediction-artifacts dataset built by `python -m src.cli package` (no `--crossed`).
   It contains `manifest.json`, `models/`, `weights.json`, and the `src/` package needed to
   unpickle the bundles. **No encoder is required for this argmax blend.**

Inference: row-normalise each bundle's margins, blend with `weights.json` (0.5/0.5) and take
the argmax per row. If an `inference.json` with `crossed: true` is present (a research-only
crossed package), the notebook fails fast. Output: `submission.parquet`, schema `id, model`.

In [ ]:
from __future__ import annotations

import glob
import json
import os
import sys
from pathlib import Path

# Kaggle inference has no internet. Force offline mode so the attached grouping
# encoder loads from disk instead of attempting a download.
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")

import pandas as pd

ARTIFACT_CANDIDATES = [
    p for p in glob.glob("/kaggle/input/**/manifest.json", recursive=True)
]
if not ARTIFACT_CANDIDATES:
    raise FileNotFoundError(
        "No manifest.json found under /kaggle/input. Attach the artifacts dataset "
        "created by `python -m src.cli package` (without --crossed)."
    )
if len(ARTIFACT_CANDIDATES) != 1:
    raise ValueError(f"expected exactly one artifact manifest, found {len(ARTIFACT_CANDIDATES)}")
ARTIFACT_DIR = Path(ARTIFACT_CANDIDATES[0]).parent
sys.path.insert(0, str(ARTIFACT_DIR))          # lets `src` be importable
sys.path.insert(0, str(ARTIFACT_DIR / "src"))  # fallback layout

from src.constants import LABELS, N_CLASSES  # noqa: E402
from src.inference import (  # noqa: E402
    load_final,
    load_weights,
    predict_dataframe,
)
from src.submission import build_submission  # noqa: E402

print(f"artifacts: {ARTIFACT_DIR}")
print(f"classes: {N_CLASSES}")

In [ ]:
TEST_PATHS = [
    Path("/kaggle/input/competitions/name-that-llm/test.csv"),
    Path("/kaggle/input/name-that-llm/test.csv"),
    Path("data/synthetic/test.csv"),  # local smoke test only
]
test_path = next((p for p in TEST_PATHS if p.is_file()), None)
if test_path is None:
    raise FileNotFoundError("could not locate test.csv in any known input path")

test = pd.read_csv(test_path)
assert {"id", "text"} <= set(test.columns), f"unexpected columns: {list(test.columns)}"
assert test["id"].is_unique, "test.csv has duplicate ids"
assert test["text"].notna().all(), "test.csv has null text"
print(f"test rows: {len(test)} from {test_path}")

In [ ]:
bundles, manifest = load_final(ARTIFACT_DIR)
assert manifest["class_order"] == list(LABELS), "artifact class order mismatch"

weights_path = ARTIFACT_DIR / "weights.json"
if not weights_path.is_file():
    raise FileNotFoundError("validated weights.json is required for submission")
weights = load_weights(weights_path)
missing = sorted(set(weights) - set(bundles))
if missing:
    raise ValueError(f"weights reference missing model bundles: {missing}")

print("models:", sorted(bundles))
print("blend weights:", weights or "equal weights")

inference_path = ARTIFACT_DIR / "inference.json"
inference = (
    json.loads(inference_path.read_text(encoding="utf-8"))
    if inference_path.is_file()
    else {"crossed": False}
)

if inference.get("crossed"):
    raise ValueError("crossed inference is withdrawn and cannot be submitted under D024")
predictions = predict_dataframe(bundles, test, weights)

assert len(predictions) == len(test)
assert set(predictions) <= set(LABELS)
# Descriptive only: hidden-test class proportions are not known.
print(pd.Series(predictions).value_counts().head())

In [ ]:
out = build_submission(
    test["id"], predictions, "submission.parquet", expected_ids=test["id"]
)
check = pd.read_parquet(out)
assert list(check.columns) == ["id", "model"]
assert len(check) == len(test)
assert check["id"].is_unique and not check.isna().any().any()
assert check["model"].isin(LABELS).all()
print(f"wrote {out} with {len(check)} rows")
check.head()

## Notes

- Internet is disabled for this competition; the notebook never fetches weights,
  models or data at runtime. `HF_HUB_OFFLINE`/`TRANSFORMERS_OFFLINE` are set so any
  attached encoder loads from disk.
- `submission.parquet` is written to the working directory, as required.
- The class order is fixed by the artifact manifest and checked against
  `src.constants.LABELS`; a mismatched artifact fails loudly instead of silently
  permuting labels.
- **Recommended path: argmax blend** (D009/D012), no encoder dependency.
- **Crossed inference (D016) is WITHDRAWN (D017/D018)** and kept for research only.
  It is label-free and transductive (clusters test passages by topic, then assigns
  one passage per model). Its apparent gain (+0.104 on the seed-7 holdout) was a
  **k=18 artifact**; at hypothetical k≈2 the paired gain is marginal
  (+0.003…+0.007, 95% CI spans 0 on the examined holdout). The seed-7 holdout is
  **examined**, not untouched. Do not submit the crossed package by default.
- The grouping encoder (research path only) is the frozen `all-MiniLM-L6-v2`
  checkpoint (Apache-2.0), attached offline; used only for clustering, never as an
  authorship feature.
- If this notebook is run on the synthetic fixture (`data/synthetic/test.csv`),
  the output is a pipeline check, not a competition submission.